# Deep Learning Mini Project  
## Simplified DRAW-style Recurrent Variational Autoencoder on Fashion-MNIST

**Student:** Md Naim Hassan Saykat  
**Program:** M1 Artificial Intelligence  
**Course:** Deep Learning  

### Project objective
This project implements a lightweight recurrent variational autoencoder inspired by the DRAW paper, using the Fashion-MNIST dataset as a computationally efficient benchmark.

### Research question
Can a simplified DRAW-style recurrent generative model produce meaningful reconstructions and samples on a small image dataset such as Fashion-MNIST?

## Imports

In [ ]:
import math
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from torchvision import datasets, transforms
from torchvision.utils import make_grid

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Create folder

In [ ]:
import os

os.makedirs("images", exist_ok=True)

## Reproducibility

In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

## Dataset

To keep the project computationally lightweight, we use the Fashion-MNIST dataset.  
It contains grayscale images of size 28×28 and is suitable for evaluating generative models under limited computational resources.

## Data loading

In [ ]:
batch_size = 128

transform = transforms.ToTensor()

train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

## Visualize samples

In [ ]:
images, labels = next(iter(train_loader))

plt.figure(figsize=(8, 8))
grid = make_grid(images[:25], nrow=5, normalize=True, pad_value=1.0)

plt.imshow(grid.permute(1, 2, 0).cpu())
plt.axis("off")
plt.title("Fashion-MNIST samples")

# Save Before Show
plt.savefig("images/fashion_mnist_samples.png", dpi=300, bbox_inches="tight")

plt.show()

## Model design

The original DRAW model uses recurrent encoder and decoder networks, together with iterative canvas updates.  
In this project, we implement a simplified DRAW-style architecture with:

- an encoder LSTM
- a decoder LSTM
- latent sampling at each recurrent step
- cumulative canvas refinement over multiple time steps

This preserves the main recurrent generation idea of DRAW while keeping the implementation manageable.

## Model definition

In [ ]:
class SimpleDRAW(nn.Module):
    def __init__(
        self,
        input_dim=784,
        enc_hidden_dim=256,
        dec_hidden_dim=256,
        latent_dim=32,
        T=8
    ):
        super().__init__()
        self.input_dim = input_dim
        self.enc_hidden_dim = enc_hidden_dim
        self.dec_hidden_dim = dec_hidden_dim
        self.latent_dim = latent_dim
        self.T = T

        # Encoder takes x and current reconstruction error information
        self.encoder_lstm = nn.LSTMCell(input_dim * 2 + dec_hidden_dim, enc_hidden_dim)

        # Latent heads
        self.fc_mu = nn.Linear(enc_hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(enc_hidden_dim, latent_dim)

        # Decoder
        self.decoder_lstm = nn.LSTMCell(latent_dim, dec_hidden_dim)

        # Write to canvas
        self.fc_write = nn.Linear(dec_hidden_dim, input_dim)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        batch_size = x.size(0)

        # Flatten input
        x = x.view(batch_size, -1)

        # Initial states
        h_enc = torch.zeros(batch_size, self.enc_hidden_dim, device=x.device)
        c_enc = torch.zeros(batch_size, self.enc_hidden_dim, device=x.device)

        h_dec = torch.zeros(batch_size, self.dec_hidden_dim, device=x.device)
        c_dec = torch.zeros(batch_size, self.dec_hidden_dim, device=x.device)

        canvas = torch.zeros(batch_size, self.input_dim, device=x.device)

        mus = []
        logvars = []

        for _ in range(self.T):
            x_hat = x - torch.sigmoid(canvas)
            enc_input = torch.cat([x, x_hat, h_dec], dim=1)

            h_enc, c_enc = self.encoder_lstm(enc_input, (h_enc, c_enc))

            mu = self.fc_mu(h_enc)
            logvar = self.fc_logvar(h_enc)
            z = self.reparameterize(mu, logvar)

            h_dec, c_dec = self.decoder_lstm(z, (h_dec, c_dec))

            canvas = canvas + self.fc_write(h_dec)

            mus.append(mu)
            logvars.append(logvar)

        x_recon = torch.sigmoid(canvas)
        return x_recon, mus, logvars

##  Instantiate model

In [ ]:
model = SimpleDRAW(
    input_dim=28 * 28,
    enc_hidden_dim=256,
    dec_hidden_dim=256,
    latent_dim=32,
    T=8
).to(device)

print(model)

## Loss function

The loss is composed of two terms:

1. **Reconstruction loss**: encourages the generated image to match the input.
2. **KL divergence**: regularizes the latent distribution toward a standard Gaussian prior.

The total loss is:

\[
\mathcal{L} = \mathcal{L}_{recon} + \beta \mathcal{L}_{KL}
\]

where \(\beta\) controls the regularization strength.

## Loss function

In [ ]:
def draw_loss(x, x_recon, mus, logvars, beta=1.0):
    x = x.view(x.size(0), -1)

    recon_loss = F.binary_cross_entropy(x_recon, x, reduction="sum")

    kl_loss = 0.0
    for mu, logvar in zip(mus, logvars):
        kl_loss = kl_loss + (-0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()))

    total_loss = recon_loss + beta * kl_loss
    return total_loss, recon_loss, kl_loss

## Optimizer

In [ ]:
learning_rate = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

## Training loop

In [ ]:
def train_one_epoch(model, dataloader, optimizer, device, beta=1.0):
    model.train()
    total_loss = 0.0
    total_recon = 0.0
    total_kl = 0.0

    for x, _ in dataloader:
        x = x.to(device)

        optimizer.zero_grad()
        x_recon, mus, logvars = model(x)
        loss, recon_loss, kl_loss = draw_loss(x, x_recon, mus, logvars, beta=beta)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()

    n = len(dataloader.dataset)
    return total_loss / n, total_recon / n, total_kl / n

## Evaluation loop

In [ ]:
def evaluate(model, dataloader, device, beta=1.0):
    model.eval()
    total_loss = 0.0
    total_recon = 0.0
    total_kl = 0.0

    with torch.no_grad():
        for x, _ in dataloader:
            x = x.to(device)
            x_recon, mus, logvars = model(x)
            loss, recon_loss, kl_loss = draw_loss(x, x_recon, mus, logvars, beta=beta)

            total_loss += loss.item()
            total_recon += recon_loss.item()
            total_kl += kl_loss.item()

    n = len(dataloader.dataset)
    return total_loss / n, total_recon / n, total_kl / n

## Experiments

## First experiment: baseline training

We first train a baseline simplified DRAW-style model with:
- latent dimension = 32
- recurrent steps = 8
- \(\beta = 1.0\)

## Train baseline

In [ ]:
epochs = 20
beta = 1.0

train_history = []
test_history = []

for epoch in range(epochs):
    train_loss, train_recon, train_kl = train_one_epoch(model, train_loader, optimizer, device, beta=beta)
    test_loss, test_recon, test_kl = evaluate(model, test_loader, device, beta=beta)

    train_history.append((train_loss, train_recon, train_kl))
    test_history.append((test_loss, test_recon, test_kl))

    print(
        f"Epoch {epoch+1:02d} | "
        f"train loss={train_loss:.4f}, recon={train_recon:.4f}, kl={train_kl:.4f} | "
        f"test loss={test_loss:.4f}, recon={test_recon:.4f}, kl={test_kl:.4f}"
    )

### Training Dynamics
The following plots show the evolution of total loss, reconstruction loss, and KL divergence during training.

## Plot losses

In [ ]:
train_loss_vals = [x[0] for x in train_history]
test_loss_vals = [x[0] for x in test_history]

train_recon_vals = [x[1] for x in train_history]
test_recon_vals = [x[1] for x in test_history]

train_kl_vals = [x[2] for x in train_history]
test_kl_vals = [x[2] for x in test_history]

plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.plot(train_loss_vals, label="Train")
plt.plot(test_loss_vals, label="Test")
plt.title("Total Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid()

plt.subplot(1, 3, 2)
plt.plot(train_recon_vals, label="Train")
plt.plot(test_recon_vals, label="Test")
plt.title("Reconstruction Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid()

plt.subplot(1, 3, 3)
plt.plot(train_kl_vals, label="Train")
plt.plot(test_kl_vals, label="Test")
plt.title("KL Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid()

plt.tight_layout()

# Save Before Show
plt.savefig("images/loss_components.png", dpi=300, bbox_inches="tight")

plt.show()

## Reconstruction visualization

In [ ]:
def show_reconstructions(model, dataloader, device, n=8, save_path="images/reconstructions.png"):
    model.eval()
    with torch.no_grad():
        x, _ = next(iter(dataloader))
        x = x[:n].to(device)

        x_recon, _, _ = model(x)

        x = x.cpu()
        x_recon = x_recon.view(-1, 1, 28, 28).cpu()

        fig = plt.figure(figsize=(2*n, 4))
        for i in range(n):
            plt.subplot(2, n, i + 1)
            plt.imshow(x[i].squeeze(), cmap="gray")
            plt.axis("off")
            if i == 0:
                plt.ylabel("Original", fontsize=12)

            plt.subplot(2, n, n + i + 1)
            plt.imshow(x_recon[i].squeeze(), cmap="gray")
            plt.axis("off")
            if i == 0:
                plt.ylabel("Recon", fontsize=12)

        plt.tight_layout()
        fig.savefig(save_path, dpi=300, bbox_inches="tight")

        plt.show()
        plt.close(fig)

## Sampling from prior

In [ ]:
def sample_from_model(model, n=16, save_path="images/generated_samples.png"):
    model.eval()
    with torch.no_grad():
        batch_size = n

        h_dec = torch.zeros(batch_size, model.dec_hidden_dim, device=device)
        c_dec = torch.zeros(batch_size, model.dec_hidden_dim, device=device)
        canvas = torch.zeros(batch_size, model.input_dim, device=device)

        for _ in range(model.T):
            z = torch.randn(batch_size, model.latent_dim, device=device)
            h_dec, c_dec = model.decoder_lstm(z, (h_dec, c_dec))
            canvas = canvas + model.fc_write(h_dec)

        samples = torch.sigmoid(canvas).view(-1, 1, 28, 28).cpu()

        fig = plt.figure(figsize=(8, 8))
        for i in range(n):
            plt.subplot(4, 4, i + 1)
            plt.imshow(samples[i].squeeze(), cmap="gray")
            plt.axis("off")

        plt.tight_layout()
        fig.savefig(save_path, dpi=300, bbox_inches="tight")

        plt.show()
        plt.close(fig)

## Second experiment: effect of latent dimension

We compare several latent dimensions to evaluate how representation capacity influences reconstruction quality and generation.

## Experiment Design

We evaluate the effect of:
- latent dimension (representation capacity)
- KL regularization (beta)

These experiments help understand how the latent space influences reconstruction and generation quality.

## Experiment helper

In [ ]:
def run_experiment(latent_dim=16, T=8, beta=1.0, epochs=5):
    exp_model = SimpleDRAW(
        input_dim=28 * 28,
        enc_hidden_dim=256,
        dec_hidden_dim=256,
        latent_dim=latent_dim,
        T=T
    ).to(device)

    exp_optimizer = torch.optim.Adam(exp_model.parameters(), lr=1e-3)

    train_hist = []
    test_hist = []

    for epoch in range(epochs):
        train_metrics = train_one_epoch(exp_model, train_loader, exp_optimizer, device, beta=beta)
        test_metrics = evaluate(exp_model, test_loader, device, beta=beta)

        train_hist.append(train_metrics)
        test_hist.append(test_metrics)

    return exp_model, train_hist, test_hist

## Run latent-dimension comparison

In [ ]:
latent_dims = [8, 16, 32]
results = {}

for ld in latent_dims:
    print(f"Running experiment with latent_dim={ld}")
    exp_model, train_hist, test_hist = run_experiment(latent_dim=ld, T=8, beta=1.0, epochs=5)
    results[ld] = {
        "model": exp_model,
        "train_hist": train_hist,
        "test_hist": test_hist
    }

## Experimental Results

## Compare final losses

In [ ]:
print("Final test losses by latent dimension:")
for ld in latent_dims:
    final_test_loss = results[ld]["test_hist"][-1][0]
    final_test_recon = results[ld]["test_hist"][-1][1]
    final_test_kl = results[ld]["test_hist"][-1][2]

    print(
        f"latent_dim={ld:2d} | "
        f"test loss={final_test_loss:.4f} | "
        f"recon={final_test_recon:.4f} | "
        f"kl={final_test_kl:.4f}"
    )

## Third experiment: effect of KL regularization

We vary the \(\beta\) coefficient in the loss to study the trade-off between reconstruction quality and latent regularization.

## Run beta comparison

In [ ]:
betas = [0.5, 1.0, 2.0]
beta_results = {}

for b in betas:
    print(f"Running experiment with beta={b}")
    exp_model, train_hist, test_hist = run_experiment(latent_dim=16, T=8, beta=b, epochs=5)
    beta_results[b] = {
        "model": exp_model,
        "train_hist": train_hist,
        "test_hist": test_hist
    }

## Compare beta results

In [ ]:
print("Final test losses by beta:")
for b in betas:
    final_test_loss = beta_results[b]["test_hist"][-1][0]
    final_test_recon = beta_results[b]["test_hist"][-1][1]
    final_test_kl = beta_results[b]["test_hist"][-1][2]

    print(
        f"beta={b:3.1f} | "
        f"test loss={final_test_loss:.4f} | "
        f"recon={final_test_recon:.4f} | "
        f"kl={final_test_kl:.4f}"
    )

## Discussion

The simplified DRAW-style model is able to reconstruct Fashion-MNIST images and generate coherent samples from the latent prior.  
The recurrent canvas refinement mechanism allows the model to progressively improve the generated image over several decoding steps.

The experiments also show that:
- increasing the latent dimension generally improves reconstruction quality,
- the KL weight influences the balance between generation regularity and reconstruction accuracy,
- a lightweight recurrent generative model can already capture meaningful structure on a small image dataset.

## Final results table

In [ ]:
import pandas as pd

rows = []

for ld in latent_dims:
    rows.append({
        "Experiment": f"latent_dim={ld}",
        "Test Loss": results[ld]["test_hist"][-1][0],
        "Recon Loss": results[ld]["test_hist"][-1][1],
        "KL Loss": results[ld]["test_hist"][-1][2],
    })

for b in betas:
    rows.append({
        "Experiment": f"beta={b}",
        "Test Loss": beta_results[b]["test_hist"][-1][0],
        "Recon Loss": beta_results[b]["test_hist"][-1][1],
        "KL Loss": beta_results[b]["test_hist"][-1][2],
    })

df_results = pd.DataFrame(rows)
df_results

## Results Analysis

Increasing the latent dimension improves reconstruction quality because the model can encode more information. However, larger latent spaces may reduce regularization and lead to less structured representations.

Varying the beta parameter shows a trade-off:
- lower beta improves reconstruction
- higher beta enforces stronger regularization

This demonstrates the balance between reconstruction accuracy and latent space structure.

## Limitations

This implementation is a simplified version of DRAW and does not include the original attention-based read/write mechanism. As a result, it may not fully capture fine-grained spatial dependencies.

## Conclusion

This project implemented a simplified recurrent generative model inspired by the DRAW paper on Fashion-MNIST.  
The model successfully learned to reconstruct and generate visually meaningful images while remaining lightweight enough for modest computational resources.

The experiments demonstrated that latent dimension and KL regularization both affect model behavior.  
Future improvements could include:
- a closer implementation of the original attention-based read/write mechanism,
- larger recurrent depth,
- longer training,
- and application to more complex grayscale datasets.

## Final Model Outputs

We visualize reconstructions and generated samples using the best-performing model selected from the beta regularization experiments.

In [ ]:
best_model = beta_results[1.0]["model"]

show_reconstructions(
    best_model,
    test_loader,
    device,
    n=8,
    save_path="images/best_model_reconstructions.png"
)

sample_from_model(
    best_model,
    n=16,
    save_path="images/best_model_generated_samples.png"
)

### Observations

The model is able to reconstruct input images with good structural fidelity, although some blurriness remains due to the variational bottleneck. Generated samples exhibit meaningful patterns, indicating that the latent space captures important features of the dataset. Increasing the latent dimension improves reconstruction quality, while the beta parameter controls the trade-off between reconstruction and regularization.